In [ ]:
%run ./imports.py

## Paths

In [ ]:
shapefile_path = "data/ne_countries/ne_10m_admin_0_countries.shp"
countries_naturalearth_path = "data/countries_naturalearth.pkl"
countries_geoloc_path = "data/countries_geolocation.csv"
min_intercountry_distances_path = "data/country_inter_min_distances.csv"
max_intracountry_distances_path = "data/country_intra_max_distances.csv"
minimal_deviation_path = "data/minimal_deviation.csv"
countries_convex_hull_path = "data/countries_convex_hull_path.pkl"

# json_sampled_path = "data/country_sampled_coordinates.json"
# json_sampled_mainland_path = "data/country_mainland_sampled_coordinates.json"
# json_sampled_shifted_path = "data/country_shifted_coordinates.json"
# json_sampled_shifted_mainland_path = "data/country_mainland_shifted_coordinates.json"

# json_unsampled_path = "data/country_unsampled_coordinates.json"
# json_unsampled_mainland_path = "data/country_mainland_unsampled_coordinates.json"
# json_unsampled_shifted_path = "data/country_unsampled_shifted_coordinates.json"
# json_unsampled_shifted_mainland_path = "data/country_mainland_unsampled_shifted_coordinates.json"

json_unsampled_path_100 = "data/country_unsampled_coordinates_100.json"
json_unsampled_mainland_path_100 = "data/country_mainland_unsampled_coordinates_100.json"
json_unsampled_shifted_path_100 = "data/country_unsampled_shifted_coordinates_100.json"
json_unsampled_shifted_mainland_path_100 = "data/country_mainland_unsampled_shifted_coordinates_100.json"

json_unsampled_path_11 = "data/country_unsampled_coordinates_11.json"
json_unsampled_mainland_path_11 = "data/country_mainland_unsampled_coordinates_11.json"
json_unsampled_shifted_path_11 = "data/country_unsampled_shifted_coordinates_11.json"
json_unsampled_shifted_mainland_path_11 = "data/country_mainland_unsampled_shifted_coordinates_11.json"

json_hull_path = "data/country_hull_coordinates.json"
json_hull_mainland_path = "data/country_hull_mainland_coordinates.json"
json_hull_shifted_path = "data/country_hull_shifted_coordinates.json"
json_hull_mainland_shifted_path = "data/country_hull_mainland_shifted_coordinates.json"

# max_distance_path = "data/max_intracountry_distances.csv"
# max_distance_mainland_path = "data/max_intracountry_mainland_distances.csv"
max_distance_exact_path = "data/max_intracountry_exact_distances.csv"
max_distance_mainland_exact_path = "data/max_intracountry_mainland_exact_distances.csv"
max_distance_hull_path = "data/max_intracountry_hull_distances.csv"
max_distance_hull_mainland_path = "data/max_intracountry_hull_mainland_distances.csv"
max_intra_hull_df_path = "data/max_intra_hull_df.pkl"
max_intra_hull_mainland_df_path = "data/max_intra_hull_mainland_df.pkl"

# min_distance_path = "data/min_intercountry_distances.csv"
# min_distance_mainland_path = "data/min_intercountry_mainland_distances.csv"
min_distance_exact_path = "data/min_intercountry_exact_distances.csv"
min_distance_mainland_exact_path = "data/min_intercountry_mainland_exact_distances.csv"
min_distance_hull_path = "data/min_intercountry_hull_distances.csv"
min_distance_hull_mainland_path = "data/min_intercountry_hull_mainland_distances.csv"
min_inter_hull_df_path = "data/min_inter_hull_df.pkl"
min_inter_hull_mainland_df_path = "data/min_inter_hull_mainland_df.pkl"

# deviated_distance_path = "data/min_deviated_distances.csv"
# deviated_distance_mainland_path = "data/min_deviated_mainland_distances.csv"
deviated_distance_exact_path = "data/min_deviated_exact_distances.csv"
deviated_distance_mainland_exact_path = "data/min_deviated_mainland_exact_distances.csv"
deviated_distance_hull_path = "data/min_deviated_hull_distances.csv"
deviated_distance_hull_mainland_path = "data/min_deviated_hull_mainland_distances.csv"
min_dev_hull_df_path = "data/min_dev_hull_df.pkl"
min_dev_hull_mainland_df_path = "data/min_dev_hull_mainland_df.pkl"

In [ ]:
C_OPTICAL_FIBER_KM_PER_MS = (2/3) * (299792458 / 10**6)

## Load data from the Natural Earth Admin 0 Shapefile

In [ ]:
gdf = gpd.read_file(shapefile_path).to_crs("EPSG:4326")

In [ ]:
gdf_relevant = gdf[['ADMIN', 'ISO_A3', 'CONTINENT', 'geometry']].rename(
    columns={'ADMIN': 'Country', 'ISO_A3': 'ISO', 'CONTINENT': 'Continent', 'geometry': 'Geometry'})
gdf_relevant['Country'] = gdf_relevant.Country.apply(lambda c: unicodedata.normalize('NFKD', c).encode('ascii', 'ignore').decode('utf-8'))
gdf_relevant = gdf_relevant.sort_values(by='Country', ascending=True).reset_index(drop=True)
gdf_relevant.head()

### Compute granularity

In [ ]:
def measure_boundary_granularity(df, geom_col="Geometry"):
    """Measure granularity: distances between consecutive exterior points."""
    all_count_points   = 0
    all_min_spacing_km = 10000
    all_max_spacing_km = 0
    all_avg_spacing_km = 0
    all_avg_median_km  = 0
    
    for c, country in enumerate(sorted(df["Country"].tolist())):
        geom = df[df["Country"] == country][geom_col].iloc[0]
        if isinstance(geom, Polygon):
            lines = [geom.exterior]
        elif isinstance(geom, MultiPolygon):
            lines = [poly.exterior for poly in geom.geoms]
        else:
            raise ValueError("Geometry must be Polygon or MultiPolygon")

        distances = []
        distances_for_median = []
        for line in lines:
            coords = list(line.coords)
            for i in range(len(coords) - 1):
                p1 = (coords[i][1], coords[i][0])  # (lat, lon)
                p2 = (coords[i + 1][1], coords[i + 1][0])
                d = geodesic(p1, p2).kilometers
                distances.append(d)
            if sum(distances) >= 1000:
                distances_for_median.extend(distances)

        # Current country
        count_points   = sum(len(line.coords) for line in lines)
        min_spacing_km = min(distances) if distances else None
        max_spacing_km = max(distances) if distances else None
        avg_spacing_km = sum(distances) / len(distances) if distances else None
        median_km      = np.median(distances_for_median) if distances_for_median else None

        # All countries
        all_min_spacing_km = min(all_min_spacing_km, min_spacing_km)
        all_max_spacing_km = max(all_max_spacing_km, max_spacing_km)
        all_avg_spacing_km = ((all_avg_spacing_km * all_count_points) + sum(distances)) / (all_count_points + len(distances))
        if median_km:
            all_avg_median_km  = ((all_avg_median_km * c) + median_km) / (c + 1)
        all_count_points += count_points

        if country in ["United States of America", "Russia", "India", "Germany", "Australia"]:
            print(f"Country: {country}")
            print(f"No. of points: {count_points}")
            print(f"Min. spacing km: {mu.rnd(min_spacing_km, 2)}")
            print(f"Max. spacing km: {mu.rnd(max_spacing_km, 2)}")
            print(f"Avg. spacing km: {mu.rnd(avg_spacing_km, 2)}")
            print(f"Avg. med. spacing km (>= 1000 km total): {mu.rnd(median_km, 2)}\n")

    return all_count_points, mu.rnd(all_min_spacing_km, 2), mu.rnd(all_max_spacing_km, 2), mu.rnd(all_avg_spacing_km, 2), mu.rnd(all_avg_median_km, 2)

In [ ]:
print(f"All countries: {measure_boundary_granularity(gdf_relevant)}")

## Sort multipolygons in descending order of area

In [ ]:
def sort_multipolygon_by_area_desc(geom):
    geod = Geod(ellps="WGS84")
    if isinstance(geom, MultiPolygon):
        sorted_polys = sorted(geom.geoms, key=lambda p: abs(geod.geometry_area_perimeter(p)[0]), reverse=True)
        return MultiPolygon(sorted_polys)
    else:
        return geom

In [ ]:
gdf_sorted = gdf_relevant.copy()
gdf_sorted['Geometry'] = gdf_sorted['Geometry'].apply(sort_multipolygon_by_area_desc)
gdf_sorted.head()

## Enrich with smallest geometry

#### Compute center

In [ ]:
def find_center_lon(geometries):
    polygons = []
    for geometry in geometries:
        if isinstance(geometry, MultiPolygon):
            polygons.append(geometry.geoms[0])
        elif isinstance(geometry, Polygon):
            polygons.append(geometry)
    max_poly = max(polygons, key=lambda p: p.area, default=None)
    if max_poly is not None:
        return max_poly.centroid.x
    return None

#### Handle longitude wraparound

In [ ]:
def lon_wraparound(lon):
    if lon > 180:
        lon -= 360
    elif lon < -180:
        lon += 360
    return lon

#### Recenter geometry

In [ ]:
def recenter_longitude(lon, center):
    return lon_wraparound(lon - center)

In [ ]:
def recenter_coords_lonlat(coords, center):
    return [(recenter_longitude(c[0], center), c[1]) for c in coords]

In [ ]:
def recenter_coords_latlon(coords, center):
    return [(c[0], recenter_longitude(c[1], center)) for c in coords]

In [ ]:
def recenter_polygon(polygon, center):
    exterior = LinearRing(recenter_coords_lonlat(polygon.exterior.coords, center))
    interiors = [LinearRing(recenter_coords_lonlat(interior.coords, center)) for interior in polygon.interiors]
    return Polygon(exterior, interiors)

In [ ]:
def recenter_geometry(geometry, center):
    if isinstance(geometry, Polygon):
        return recenter_polygon(geometry, center)
    if isinstance(geometry, MultiPolygon):
        return MultiPolygon([recenter_polygon(geom, center) for geom in geometry.geoms])
    return None

#### Restore polygons

In [ ]:
def restore_longitude(lon, center):
    return lon_wraparound(lon + center)

In [ ]:
def restore_coords_lonlat(coords, center):
    return [(restore_longitude(c[0], center), c[1]) for c in coords]

In [ ]:
def restore_coords_latlon(coords, center):
    return [(c[0], restore_longitude(c[1], center)) for c in coords]

#### Polygon to coordinates

In [ ]:
def poly_latlon(polygon):
    return [(lat, lon) for lon, lat in polygon.exterior.coords]

#### Smallest geometries (in cartesian coordinates)

In [ ]:
def compute_smallest_geometry_mainland(geometry):
    # Find center latitide
    lon_center = find_center_lon([geometry])

    if isinstance(geometry, MultiPolygon):
        polygon = geometry.geoms[0]
    elif isinstance(geometry, Polygon):
        polygon = geometry
    else:
        print("Error: Not polygonal")
        return None

    polygon_recentered = recenter_polygon(polygon, lon_center)
    return translate(polygon_recentered, xoff=lon_center)

In [ ]:
def compute_smallest_geometry(geometry):
    # Find center latitude
    lon_center = find_center_lon([geometry])
    
    if isinstance(geometry, MultiPolygon):
        polygons = []
        for poly in geometry.geoms:
            recentered_polygon = recenter_polygon(poly, lon_center)
            polygons.append(translate(recentered_polygon, xoff=lon_center))
        return MultiPolygon(polygons)
        
    if isinstance(geometry, Polygon):
        recentered_polygon = recenter_polygon(geometry, lon_center)
        return translate(recentered_polygon, xoff=lon_center)
    
    print("Error: Not polygonal for country")
    return None

#### Shift geometries

In [ ]:
def compute_shifted_geometry(geometry):
    return [translate(geometry, xoff=lon_offset)
                for lon_offset in [0, 360, -360]]

#### Enrich geometry

In [ ]:
gdf_geometry_enriched = gdf_sorted.copy()
# Add smallest geometry
gdf_geometry_enriched['Smallest_Geometry'] = gdf_geometry_enriched.Geometry.parallel_apply(
    compute_smallest_geometry)
gdf_geometry_enriched['Smallest_Geometry_Mainland'] = gdf_geometry_enriched.Geometry.parallel_apply(
    compute_smallest_geometry_mainland)
gdf_geometry_enriched['Smallest_Geometry_shifted'] = gdf_geometry_enriched['Smallest_Geometry'].parallel_apply(
    compute_shifted_geometry)
gdf_geometry_enriched['Smallest_Geometry_Mainland_shifted'] = gdf_geometry_enriched['Smallest_Geometry_Mainland'].parallel_apply(
    compute_shifted_geometry)
# Print head
gdf_geometry_enriched.head(n=1)

#### Plot country mainland centroids

In [ ]:
centroids = [poly.centroid for poly in gdf_geometry_enriched["Smallest_Geometry_Mainland"].tolist()]
centroids_latlon = [(c.y, c.x) for c in centroids]
m = pu.folium_points(centroids_latlon, {"icon_color": "orange"})
m

#### Downsample geometries

In [ ]:
def downsample_boundary(geom, target_spacing_km=25):
    if isinstance(geom, Polygon):
        rings = [geom.exterior]
    elif isinstance(geom, MultiPolygon):
        rings = [poly.exterior for poly in geom.geoms]
    else:
        raise ValueError("Geometry must be Polygon or MultiPolygon")

    downsampled_rings = []
    for ring in rings:
        coords = list(ring.coords)
        if len(coords) < 1:
            continue
            
        new_coords = [coords[0]]  # Always keep the first point
        last_point = coords[0]

        for pt in coords[1:]:
            dist = geodesic((last_point[1], last_point[0]), (pt[1], pt[0])).kilometers
            if dist >= target_spacing_km:
                new_coords.append(pt)
                last_point = pt  # Update last kept point

        # Ensure closed ring (last point = first point)
        if new_coords[0] != new_coords[-1]:
            new_coords.append(new_coords[0])

        # Check minimum required points for a LinearRing
        if len(new_coords) >= 4:
            downsampled_rings.append(LinearRing(new_coords))
        else:
            downsampled_rings.append(LinearRing(coords))  # Fallback to original full ring

    if isinstance(geom, Polygon):
        return Polygon(downsampled_rings[0])
    else:
        return MultiPolygon([Polygon(ring) for ring in downsampled_rings])

### Test

#### Plot smallest geometry

In [ ]:
geod = Geod(ellps="WGS84")

def sample_geometry_n_coords(polygon, n):
    random.seed(42)
    coords = polygon.exterior.coords
    area = abs(geod.geometry_area_perimeter(polygon)[0])
    if area < 1000:
        n = 1
    if len(coords) > n:
        idxs_sampled = sorted(random.sample(range(len(coords)), n))
        idxs_sampled += [idxs_sampled[0]]
        coords_sampled = [coords[i] for i in idxs_sampled]
    else:
        coords_sampled = coords
    return [(lat, lon) for lon, lat in coords_sampled]

#### Print row

In [ ]:
country = 'United States of America'
gdf_geometry_enriched[gdf_geometry_enriched['Country'] == country].head()

In [ ]:
orig_geom = gdf_geometry_enriched[gdf_geometry_enriched['Country'] == country]['Geometry'].iloc[0]
small_geom = gdf_geometry_enriched[gdf_geometry_enriched['Country'] == country]['Smallest_Geometry'].iloc[0]
# small_sampled_geom = gdf_geometry_enriched[gdf_geometry_enriched['Country'] == country]['Smallest_Geometry_sampled'].iloc[0]
small_shifted_geom = gdf_geometry_enriched[gdf_geometry_enriched['Country'] == country]['Smallest_Geometry_shifted'].iloc[0]

In [ ]:
m = pu.folium_points([(0, 0)], {"icon_color": "white"})
for i, g in enumerate(orig_geom.geoms):
    m = pu.folium_add_polyline(m, sample_geometry_n_coords(g, 20))
    if i == 1: break
for i, g in enumerate(small_geom.geoms):
    m = pu.folium_add_polyline(m, sample_geometry_n_coords(g, 20), {"icon_color": "orange"})
    if i == 1: break
# for i, g in enumerate(small_sampled_geom.geoms):
#     m = pu.folium_add_polyline(m, sample_geometry_n_coords(g, 20), {"icon_color": "green"})
#     if i == 1: break
for shifted_g in small_shifted_geom:
    for i, g in enumerate(shifted_g.geoms):
        m = pu.folium_add_polyline(m, sample_geometry_n_coords(g, 20), {"icon_color": "purple", "line_color": "purple"})
        if i == 1: break
m = pu.folium_add_meridians(m)
m

In [ ]:
# print(measure_boundary_granularity(gdf_geometry_enriched, "Smallest_Geometry_sampled"))

### Save

In [ ]:
def to_latlon(lonlat):
    if isinstance(lonlat, tuple):
        return (lonlat[1], lonlat[0])
    elif isinstance(lonlat, list):
        return [to_latlon(p) for p in lonlat]

In [ ]:
def multipoly_latlon(geom):
    coords = []
    if isinstance(geom, Polygon):
        coords = poly_latlon(geom)
    elif isinstance(geom, MultiPolygon):
        for g in geom.geoms:
            coords.extend(poly_latlon(g))
    else:
        raise ValueError("Geometry must be Polygon or MultiPolygon")
    return coords

In [ ]:
def save_as_json(df, col_name, json_path, num=0):
    coords = {}
    countries = df["Country"].tolist()
    if num == 0 or num >= len(countries):
        selected_countries = countries
    else:
        random.seed(0)
        selected_countries = sorted(random.sample(countries, num))
    for country in selected_countries:
        coords[country] = []
        if "_shifted" in col_name:
            geoms = df[df["Country"] == country][col_name].iloc[0]
            for geom in geoms:
                coords[country].append(multipoly_latlon(geom))
        else:
            geom = df[df["Country"] == country][col_name].iloc[0]
            coords[country] = multipoly_latlon(geom)
    
    with open(json_path, "w") as fp:
        json.dump(coords, fp)

In [ ]:
# save_as_json(gdf_geometry_enriched, "Smallest_Geometry_sampled", json_sampled_path)
# save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland_sampled", json_sampled_mainland_path)
# save_as_json(gdf_geometry_enriched, "Smallest_Geometry_sampled_shifted", json_sampled_shifted_path)
# save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland_sampled_shifted", json_sampled_shifted_mainland_path)

save_as_json(gdf_geometry_enriched, "Smallest_Geometry", json_unsampled_path_100, 100)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland", json_unsampled_mainland_path_100, 100)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_shifted", json_unsampled_shifted_path_100, 100)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland_shifted", json_unsampled_shifted_mainland_path_100, 100)

save_as_json(gdf_geometry_enriched, "Smallest_Geometry", json_unsampled_path_11, 11)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland", json_unsampled_mainland_path_11, 11)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_shifted", json_unsampled_shifted_path_11, 11)
save_as_json(gdf_geometry_enriched, "Smallest_Geometry_Mainland_shifted", json_unsampled_shifted_mainland_path_11, 11)

In [ ]:
gdf_geometry_enriched.to_pickle(countries_naturalearth_path)

## Check approximation error when convex hulls are used

### Maximum intra-country distance

#### Maximum intra-country distance: unsampled coordinates

In [ ]:
%%bash -s "$json_unsampled_mainland_path_100" "$max_distance_mainland_exact_path"
./cpp/geolocation_based_analysis/BUILD/max_intracountry_distance "$1" "$2"

#### Maximum intra-country distance: convex hull

In [ ]:
def compute_max_intracountry_distances(df, num=100):
    # Select 100 countries
    random.seed(0)
    selected_countries = sorted(random.sample(df["Country"].tolist(), num))
    max_dists = []
    
    for country in selected_countries:
        geometry = df[df["Country"] == country]["Smallest_Geometry_Mainland"].iloc[0]
        coords = to_latlon(list(geometry.convex_hull.exterior.coords))
        
        max_dist = 0
        p1_max = (0, 0)
        p2_max = (0, 0)
        
        for p1, p2 in combinations(coords, 2):
            dist = geodesic(p1, p2).km
            if dist > max_dist:
                max_dist = dist
                p1_max = p1
                p2_max = p2

        max_dists.append((country, round(max_dist, 2), p1_max[0], p1_max[1], p2_max[0], p2_max[1]))

    return max_dists

In [ ]:
max_dists_approx = compute_max_intracountry_distances(gdf_geometry_enriched, 100)

#### Maximum intra-country distance: Compare two methods

In [ ]:
df_maxdist_exact = pd.read_csv(max_distance_mainland_exact_path)
df_maxdist_exact[df_maxdist_exact["Country"] == "United States of America"].head()

In [ ]:
df_maxdist_approx = pd.DataFrame(max_dists_approx, columns=["Country", "MaxDistance_km", "Lat1", "Lon1", "Lat2", "Lon2"])
df_maxdist_approx[df_maxdist_approx["Country"] == "United States of America"].head()

In [ ]:
maxdists_exact = df_maxdist_exact["MaxDistance_km"].tolist()
maxdists_approx = df_maxdist_approx["MaxDistance_km"].tolist()
err_abs = []
err_pct = []
for ex, ap in zip(maxdists_exact, maxdists_approx):
    err_abs.append(abs(ex - ap))
    err_pct.append(abs(ex - ap) * 100.0 / ap)
print(mu.pctile_rnd(err_abs, 25, 2), mu.pctile_rnd(err_abs, 50, 2), mu.pctile_rnd(err_abs, 75, 2))
print(mu.pctile_rnd(err_pct, 25, 2), mu.pctile_rnd(err_pct, 50, 2), mu.pctile_rnd(err_pct, 75, 2))

### Minimum inter-country distance

#### Minimum inter-country distance: unsampled coordinates

In [ ]:
%%bash -s "$json_unsampled_shifted_mainland_path_11" "$min_distance_mainland_exact_path"
./cpp/geolocation_based_analysis/BUILD/min_intercountry_distance "$1" "$2"

#### Minimum inter-country distance: convex hull

In [ ]:
def compute_min_intercountry_distances(df, num=11):
    # Select 11 countries
    random.seed(0)
    selected_countries = sorted(random.sample(df["Country"].tolist(), num))
    min_dists = []
    
    for country1, country2 in combinations(selected_countries, 2):
        geometry1 = df[df["Country"] == country1]["Smallest_Geometry_Mainland"].iloc[0]
        geometry2 = df[df["Country"] == country2]["Smallest_Geometry_Mainland"].iloc[0]
        coords1 = to_latlon(list(geometry1.convex_hull.exterior.coords))
        coords2 = to_latlon(list(geometry2.convex_hull.exterior.coords))
        
        min_dist = np.inf
        p1_min = (0, 0)
        p2_min = (0, 0)
        
        for p1 in coords1:
            for p2 in coords2:
                dist = geodesic(p1, p2).km
                if dist < min_dist:
                    min_dist = dist
                    p1_min = p1
                    p2_min = p2

        min_dists.append((country1, country2, round(min_dist, 2), p1_min[0], p1_min[1], p2_min[0], p2_min[1]))
        min_dists.append((country2, country1, round(min_dist, 2), p2_min[0], p2_min[1], p1_min[0], p1_min[1]))

    return min_dists

In [ ]:
min_dists_approx = compute_min_intercountry_distances(gdf_geometry_enriched, 11)

#### Minimum inter-country distance: Compare two methods

In [ ]:
df_mindist_exact = pd.read_csv(min_distance_mainland_exact_path)
df_mindist_exact = df_mindist_exact.sort_values(by=['Country1','Country2'])
df_mindist_exact.head(n=1)

In [ ]:
df_mindist_approx = pd.DataFrame(min_dists_approx, columns=["Country1", "Country2", "MinDistance_km", "Lat1", "Lon1", "Lat2", "Lon2"])
df_mindist_approx = df_mindist_approx.sort_values(by=['Country1','Country2'])
df_mindist_approx.head(n=1)

In [ ]:
random.seed(0)
selected_countries = sorted(random.sample(gdf_geometry_enriched["Country"].tolist(), 11))

err_abs = []
err_pct = []

for country1, country2 in combinations(selected_countries, 2):
    ex = df_mindist_exact[(df_mindist_exact["Country1"] == country1) & (df_mindist_exact["Country2"] == country2)]["MinDistance_km"].iloc[0]
    ap = df_mindist_approx[(df_mindist_approx["Country1"] == country1) & (df_mindist_approx["Country2"] == country2)]["MinDistance_km"].iloc[0]
    err_abs.append(abs(ex - ap))
    err_pct.append(abs(ex - ap) * 100.0 / ap)
    
print(mu.pctile_rnd(err_abs, 25, 5), mu.pctile_rnd(err_abs, 50, 5), mu.pctile_rnd(err_abs, 75, 5))
print(mu.pctile_rnd(err_pct, 25, 5), mu.pctile_rnd(err_pct, 50, 5), mu.pctile_rnd(err_pct, 75, 5))

### Minimum deviation

#### Minimum deviation: unsampled coordinates

In [ ]:
%%bash -s "$json_unsampled_shifted_mainland_path_11" "$deviated_distance_mainland_exact_path"
./cpp/geolocation_based_analysis/BUILD/min_deviated_distance "$1" "$2"

#### Minimum deviation: convex hull

In [ ]:
def compute_min_triplets(df, num=11):
    # Select 11 countries
    random.seed(0)
    selected_countries = sorted(random.sample(df["Country"].tolist(), num))
    min_dists = []
    done_count = 0
    
    for country1 in selected_countries:
        for country2 in selected_countries:
            if country1 == country2:
                continue

            geometry1 = df[df["Country"] == country1]["Smallest_Geometry_Mainland"].iloc[0]
            geometry2 = df[df["Country"] == country2]["Smallest_Geometry_Mainland"].iloc[0]
            coords1 = to_latlon(list(geometry1.convex_hull.exterior.coords))
            coords2 = to_latlon(list(geometry2.convex_hull.exterior.coords))
            
            min_dist = np.inf
            p1_min = (0, 0)
            p2_min = (0, 0)
            p3_min = (0, 0)
            
            for p1 in coords1:
                for p2 in coords1:
                    d_p1p2 = geodesic(p1, p2).km
                    for p3 in coords2:
                        dist = geodesic(p1, p3).km + geodesic(p2, p3).km - d_p1p2
                        if dist < min_dist:
                            min_dist = dist
                            p1_min = p1
                            p2_min = p2
                            p3_min = p3
    
            min_dists.append((country1, country2, round(min_dist, 5), p1_min[0], p1_min[1], p2_min[0], p2_min[1], p3_min[0], p3_min[1]))
            done_count += 1
            print(f"{done_count} country-pairs out of {num * (num - 1)}: {country1} attacked from {country2}")

    return min_dists

In [ ]:
min_devs_approx = compute_min_triplets(gdf_geometry_enriched, 11)

#### Minimum deviation: compare two methods

In [ ]:
df_mindev_exact = pd.read_csv(deviated_distance_mainland_exact_path)
df_mindev_exact = df_mindev_exact.sort_values(by=['Country1','Country2'])
df_mindev_exact.head(n=1)

In [ ]:
df_mindev_approx = pd.DataFrame(min_devs_approx, columns=["Country1", "Country2", "MinDistance_km", "S_lat", "S_lon", "D_lat", "D_lon", "A_lat", "A_lon"])
df_mindev_approx = df_mindev_approx.sort_values(by=['Country1','Country2'])
df_mindev_approx.head(n=1)

In [ ]:
random.seed(0)
selected_countries = sorted(random.sample(gdf_geometry_enriched["Country"].tolist(), 11))

err_abs = []
err_pct = []

for country1 in selected_countries:
    for country2 in selected_countries:
        if country1 == country2:
            continue
            
        ex = df_mindev_exact[(df_mindev_exact["Country1"] == country1) & (df_mindev_exact["Country2"] == country2)]["MinDistance_km"].iloc[0]
        ap = df_mindev_approx[(df_mindev_approx["Country1"] == country1) & (df_mindev_approx["Country2"] == country2)]["MinDistance_km"].iloc[0]
        err_abs.append(abs(ex - ap))
        err_pct.append(abs(ex - ap) * 100.0 / ap)
    
print(mu.pctile_rnd(err_abs, 25, 5), mu.pctile_rnd(err_abs, 50, 5), mu.pctile_rnd(err_abs, 75, 5))
print(mu.pctile_rnd(err_pct, 25, 5), mu.pctile_rnd(err_pct, 50, 5), mu.pctile_rnd(err_pct, 75, 5))

## Enrich countries with convex hull coordinates

In [ ]:
gdf_convex_hull = gdf_geometry_enriched.copy()
gdf_convex_hull["Smallest_Hull_latlon"] = gdf_convex_hull["Smallest_Geometry"].parallel_apply(lambda x: poly_latlon(x.convex_hull))
gdf_convex_hull["Smallest_Hull_Mainland_latlon"] = gdf_convex_hull["Smallest_Geometry_Mainland"].parallel_apply(lambda x: poly_latlon(x.convex_hull))
gdf_convex_hull["Smallest_Hull_shifted_latlon"] = gdf_convex_hull["Smallest_Geometry"].parallel_apply(
    lambda geom: [poly_latlon(g) for g in compute_shifted_geometry(geom.convex_hull)])
gdf_convex_hull["Smallest_Hull_Mainland_shifted_latlon"] = gdf_convex_hull["Smallest_Geometry_Mainland"].parallel_apply(
    lambda geom: [poly_latlon(g) for g in compute_shifted_geometry(geom.convex_hull)])
gdf_convex_hull = gdf_convex_hull[[
    "Country", "Continent",
    "Smallest_Hull_latlon", "Smallest_Hull_Mainland_latlon",
    "Smallest_Hull_shifted_latlon", "Smallest_Hull_Mainland_shifted_latlon"]].copy()
gdf_convex_hull.head(n=1)

In [ ]:
gdf_convex_hull.to_pickle(countries_convex_hull_path)

### Test

In [ ]:
m = pu.folium_meridians()
for country in ["United States of America", "China"]:
    m = pu.folium_add_polyline(m, gdf_convex_hull[gdf_convex_hull["Country"] == country]["Smallest_Hull_latlon"].iloc[0])
    m = pu.folium_add_polyline(m, gdf_convex_hull[gdf_convex_hull["Country"] == country]["Smallest_Hull_Mainland_latlon"].iloc[0],
                              {"line_color": "orange"})
    for geom in gdf_convex_hull[gdf_convex_hull["Country"] == country]["Smallest_Hull_shifted_latlon"].iloc[0]:
        m = pu.folium_add_polyline(m, geom)
    for geom in gdf_convex_hull[gdf_convex_hull["Country"] == country]["Smallest_Hull_Mainland_shifted_latlon"].iloc[0]:
        m = pu.folium_add_polyline(m, geom, {"line_color": "orange"})
m

### Maximum intra-country distance

#### Save input data

In [ ]:
def save_coords_as_json(df, col_name, json_path):
    coords = {}
    for country in sorted(df["Country"].tolist()):
        coords[country] = []
        if "_shifted" in col_name:
            geoms = df[df["Country"] == country][col_name].iloc[0]
            for geom in geoms:
                coords[country].append(geom)
        else:
            geom = df[df["Country"] == country][col_name].iloc[0]
            coords[country] = geom
    
    with open(json_path, "w") as fp:
        json.dump(coords, fp)

In [ ]:
save_coords_as_json(gdf_convex_hull, "Smallest_Hull_latlon", json_hull_path)
save_coords_as_json(gdf_convex_hull, "Smallest_Hull_Mainland_latlon", json_hull_mainland_path)
save_coords_as_json(gdf_convex_hull, "Smallest_Hull_shifted_latlon", json_hull_shifted_path)
save_coords_as_json(gdf_convex_hull, "Smallest_Hull_Mainland_shifted_latlon", json_hull_mainland_shifted_path)

#### Compute

In [ ]:
%%bash -s "$json_hull_path" "$max_distance_hull_path"
./cpp/geolocation_based_analysis/BUILD/max_intracountry_distance "$1" "$2"

In [ ]:
%%bash -s "$json_hull_mainland_path" "$max_distance_hull_mainland_path"
./cpp/geolocation_based_analysis/BUILD/max_intracountry_distance "$1" "$2"

In [ ]:
df_maxdist_hull = pd.read_csv(max_distance_hull_path)
df_maxdist_hull["MaxOWD_ms"] = df_maxdist_hull["MaxDistance_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_maxdist_hull.head(n=1)

In [ ]:
df_maxdist_hull_mainland = pd.read_csv(max_distance_hull_mainland_path)
df_maxdist_hull_mainland["MaxOWD_ms"] = df_maxdist_hull_mainland["MaxDistance_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_maxdist_hull_mainland.head(n=1)

In [ ]:
distances = df_maxdist_hull_mainland["MaxDistance_km"].tolist()
owds = df_maxdist_hull_mainland["MaxOWD_ms"].tolist()
for p in range(25, 99, 25):
    print(f"p{p} distance = {round(np.percentile(distances, p), 1)} km, owd = {round(np.percentile(owds, p), 1)} ms")

#### Test

In [ ]:
countries = ["United States of America", "New Zealand"]
m = pu.folium_meridians()
for country in countries:
    lat1 = df_maxdist_hull[df_maxdist_hull["Country"] == country]["Lat1"].iloc[0]
    lon1 = df_maxdist_hull[df_maxdist_hull["Country"] == country]["Lon1"].iloc[0]
    lat2 = df_maxdist_hull[df_maxdist_hull["Country"] == country]["Lat2"].iloc[0]
    lon2 = df_maxdist_hull[df_maxdist_hull["Country"] == country]["Lon2"].iloc[0]
    latm1 = df_maxdist_hull_mainland[df_maxdist_hull_mainland["Country"] == country]["Lat1"].iloc[0]
    lonm1 = df_maxdist_hull_mainland[df_maxdist_hull_mainland["Country"] == country]["Lon1"].iloc[0]
    latm2 = df_maxdist_hull_mainland[df_maxdist_hull_mainland["Country"] == country]["Lat2"].iloc[0]
    lonm2 = df_maxdist_hull_mainland[df_maxdist_hull_mainland["Country"] == country]["Lon2"].iloc[0]
    m = pu.folium_add_polyline(m, [(lat1, lon1), (lat2, lon2)], {"icons": True})
    m = pu.folium_add_polyline(m, [(latm1, lonm1), (latm2, lonm2)], {"icons": True, "icon_color": "orange", "line_color": "orange"})
m

#### Save results

In [ ]:
df_maxdist_hull.to_pickle(max_intra_hull_df_path)
df_maxdist_hull_mainland.to_pickle(max_intra_hull_mainland_df_path)

## Minimum inter-country distance for each pair of countries

#### Compute

In [ ]:
%%bash -s "$json_hull_shifted_path" "$min_distance_hull_path"
./cpp/geolocation_based_analysis/BUILD/min_intercountry_distance "$1" "$2"

In [ ]:
%%bash -s "$json_hull_mainland_shifted_path" "$min_distance_hull_mainland_path"
./cpp/geolocation_based_analysis/BUILD/min_intercountry_distance "$1" "$2"

In [ ]:
df_mindist_hull = pd.read_csv(min_distance_hull_path)
df_mindist_hull["MinOWD_ms"] = df_mindist_hull["MinDistance_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindist_hull.head(n=1)

In [ ]:
df_mindist_hull_mainland = pd.read_csv(min_distance_hull_mainland_path)
df_mindist_hull_mainland["MinOWD_ms"] = df_mindist_hull_mainland["MinDistance_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindist_hull_mainland.head(n=1)

In [ ]:
distances = df_mindist_hull_mainland["MinDistance_km"].tolist()
owds = df_mindist_hull_mainland["MinOWD_ms"].tolist()
for p in range(25, 99, 25):
    print(f"p{p} distance = {round(np.percentile(distances, p), 1)} km, owd = {round(np.percentile(owds, p), 1)} ms")

#### Test

In [ ]:
country_pairs = [("United States of America", "New Zealand"), ("France", "India")]
m = pu.folium_meridians()
for country1, country2 in country_pairs:
    lat1 = df_mindist_hull[(df_mindist_hull["Country1"] == country1) & (df_mindist_hull["Country2"] == country2)]["Lat1"].iloc[0]
    lon1 = df_mindist_hull[(df_mindist_hull["Country1"] == country1) & (df_mindist_hull["Country2"] == country2)]["Lon1"].iloc[0]
    lat2 = df_mindist_hull[(df_mindist_hull["Country1"] == country1) & (df_mindist_hull["Country2"] == country2)]["Lat2"].iloc[0]
    lon2 = df_mindist_hull[(df_mindist_hull["Country1"] == country1) & (df_mindist_hull["Country2"] == country2)]["Lon2"].iloc[0]
    latm1 = df_mindist_hull_mainland[(df_mindist_hull_mainland["Country1"] == country1) & (df_mindist_hull_mainland["Country2"] == country2)]["Lat1"].iloc[0]
    lonm1 = df_mindist_hull_mainland[(df_mindist_hull_mainland["Country1"] == country1) & (df_mindist_hull_mainland["Country2"] == country2)]["Lon1"].iloc[0]
    latm2 = df_mindist_hull_mainland[(df_mindist_hull_mainland["Country1"] == country1) & (df_mindist_hull_mainland["Country2"] == country2)]["Lat2"].iloc[0]
    lonm2 = df_mindist_hull_mainland[(df_mindist_hull_mainland["Country1"] == country1) & (df_mindist_hull_mainland["Country2"] == country2)]["Lon2"].iloc[0]
    m = pu.folium_add_polyline(m, [(lat1, lon1), (lat2, lon2)], {"icons": True})
    m = pu.folium_add_polyline(m, [(latm1, lonm1), (latm2, lonm2)], {"icons": True, "icon_color": "orange", "line_color": "orange"})
m

#### Save

In [ ]:
df_mindist_hull.to_pickle(min_inter_hull_df_path)
df_mindist_hull_mainland.to_pickle(min_inter_hull_mainland_df_path)

## Worst-case selection of source (s), destination (d) in country c1 and attacker (a) in country c2
Select s, d, a such that the difference in minimum RTT before and during attack is minimized.

#### Compute

In [ ]:
%%bash -s "$json_hull_shifted_path" "$deviated_distance_hull_path"
./cpp/geolocation_based_analysis/BUILD/min_deviated_distance "$1" "$2"

In [ ]:
%%bash -s "$json_hull_mainland_shifted_path" "$deviated_distance_hull_mainland_path"
./cpp/geolocation_based_analysis/BUILD/min_deviated_distance "$1" "$2"

In [ ]:
df_mindev_hull = pd.read_csv(deviated_distance_hull_path)
df_mindev_hull["MinDevRTT_ms"] = df_mindev_hull["MinDeviation_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull["PreAttack_ms"] = df_mindev_hull["PreAttack_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull["PostAttack_ms"] = df_mindev_hull["PostAttack_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull.head(n=1)

In [ ]:
df_mindev_hull_mainland = pd.read_csv(deviated_distance_hull_mainland_path)
df_mindev_hull_mainland["MinDevRTT_ms"] = df_mindev_hull_mainland["MinDeviation_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull_mainland["PreAttack_ms"] = df_mindev_hull_mainland["PreAttack_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull_mainland["PostAttack_ms"] = df_mindev_hull_mainland["PostAttack_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_mindev_hull_mainland.head(n=1)

#### Test

In [ ]:
country_pairs = [("United States of America", "New Zealand"), ("France", "India")]
m = pu.folium_meridians()
for country1, country2 in country_pairs:
    S_lat = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["S_lat"].iloc[0]
    S_lon = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["S_lon"].iloc[0]
    D_lat = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["D_lat"].iloc[0]
    D_lon = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["D_lon"].iloc[0]
    A_lat = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["A_lat"].iloc[0]
    A_lon = df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)]["A_lon"].iloc[0]
    S_latm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["S_lat"].iloc[0]
    S_lonm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["S_lon"].iloc[0]
    D_latm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["D_lat"].iloc[0]
    D_lonm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["D_lon"].iloc[0]
    A_latm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["A_lat"].iloc[0]
    A_lonm = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]["A_lon"].iloc[0]
    m = pu.folium_add_polyline(m, [(S_lat, S_lon), (D_lat, D_lon), (A_lat, A_lon), (S_lat, S_lon)], {"icons": True})
    m = pu.folium_add_polyline(m, [(S_latm, S_lonm), (D_latm, D_lonm), (A_latm, A_lonm), (S_latm, S_lonm)], {"icons": True, "icon_color": "orange", "line_color": "orange"})
m

#### Save

In [ ]:
df_mindev_hull.to_pickle(min_dev_hull_df_path)
df_mindev_hull_mainland.to_pickle(min_dev_hull_mainland_df_path)

## Anecdotes

In [ ]:
country1 = "United States of America"
country2 = "China"

In [ ]:
def move_left(coords):
    new_coords = []
    for lat, lon in coords:
        if lon > 0:
            new_coords.append((lat, lon - 360))
        else:
            new_coords.append((lat, lon))
    return new_coords
    
row_desired = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]
S = (row_desired["S_lat"].iloc[0], row_desired["S_lon"].iloc[0])
D = (row_desired["D_lat"].iloc[0], row_desired["D_lon"].iloc[0])
A = (row_desired["A_lat"].iloc[0], row_desired["A_lon"].iloc[0])

geod = Geod(ellps="WGS84")
interm_SD = geod.npts(S[1], S[0], D[1], D[0], 1000)
interm_SA = geod.npts(S[1], S[0], A[1], A[0], 1000)
interm_DA = geod.npts(D[1], D[0], A[1], A[0], 1000)
# Add start and end
line_coords_SD = [S] + [(lat, lon) for lon, lat in interm_SD] + [D]
line_coords_SA = move_left([S] + [(lat, lon) for lon, lat in interm_SA] + [A])
line_coords_DA = move_left([D] + [(lat, lon) for lon, lat in interm_DA] + [A])

# Plot on map
m = pu.folium_polyline(line_coords_SD, {"line_color": "green", "weight": 5})
m = pu.folium_add_polyline(m, line_coords_SA, {"line_color": "red", "weight": 5})
m = pu.folium_add_polyline(m, line_coords_DA, {"line_color": "red", "weight": 5})
m = pu.folium_add_points(m, [S, D], {"icon_color": "green"})
m = pu.folium_add_points(m, [A], {"icon_color": "red"})

m

In [ ]:
top15 = df_mindev_hull.nlargest(15, "MinDevRTT_ms")
top15.head(n=15)

In [ ]:
country1 = "United Kingdom"
country2 = "North Korea"

In [ ]:
df_mindev_hull[(df_mindev_hull["Country1"] == country1) & (df_mindev_hull["Country2"] == country2)].head()

In [ ]:
maxdist_row = df_maxdist_hull[df_maxdist_hull["Country"] == country1]
maxdist_mainland_row = df_maxdist_hull_mainland[df_maxdist_hull_mainland["Country"] == country1]
S = (maxdist_row["Lat1"].iloc[0], maxdist_row["Lon1"].iloc[0])
D = (maxdist_row["Lat2"].iloc[0], maxdist_row["Lon2"].iloc[0])
Sm = (maxdist_mainland_row["Lat1"].iloc[0], maxdist_mainland_row["Lon1"].iloc[0])
Dm = (maxdist_mainland_row["Lat2"].iloc[0], maxdist_mainland_row["Lon2"].iloc[0])
m = pu.folium_polyline([S, D], {"icons": True})
m = pu.folium_add_polyline(m, [Sm, Dm], {"icons": True, "icon_color": "orange", "line_color": "orange"})
m

In [ ]:
mindist_row = df_mindist_hull[(df_mindist_hull["Country1"] == country1) & (df_mindist_hull["Country2"] == country2)]
mindist_mainland_row = df_mindist_hull_mainland[(df_mindist_hull_mainland["Country1"] == country1) & (df_mindist_hull_mainland["Country2"] == country2)]
S = (mindist_row["Lat1"].iloc[0], mindist_row["Lon1"].iloc[0])
D = (mindist_row["Lat2"].iloc[0], mindist_row["Lon2"].iloc[0])
Sm = (mindist_mainland_row["Lat1"].iloc[0], mindist_mainland_row["Lon1"].iloc[0])
Dm = (mindist_mainland_row["Lat2"].iloc[0], mindist_mainland_row["Lon2"].iloc[0])
m = pu.folium_polyline([S, D], {"icons": True})
m = pu.folium_add_polyline(m, [Sm, Dm], {"icons": True, "icon_color": "orange", "line_color": "orange"})
m

### Plots for presentation

In [ ]:
# San Francisco: Victim (D)
# New York: Peer (S)
# Amsterdam: Attacker (A)
S = (37.773972, -122.431297)
D = (40.730610, -73.935242)
A = (52.377956, 4.897070)

m = pu.folium_points([S, D], {"icon_color": "green"})
m = pu.folium_add_points(m, [A], {"icon_color": "red"})

m

In [ ]:
country1 = "United States of America"
country2 = "Netherlands"

row_desired = df_mindev_hull_mainland[(df_mindev_hull_mainland["Country1"] == country1) & (df_mindev_hull_mainland["Country2"] == country2)]
S = (row_desired["S_lat"].iloc[0], row_desired["S_lon"].iloc[0])
D = (row_desired["D_lat"].iloc[0], row_desired["D_lon"].iloc[0])
A = (row_desired["A_lat"].iloc[0], row_desired["A_lon"].iloc[0])

geod = Geod(ellps="WGS84")
interm_SD = geod.npts(S[1], S[0], D[1], D[0], 1000)
interm_SA = geod.npts(S[1], S[0], A[1], A[0], 1000)
interm_DA = geod.npts(D[1], D[0], A[1], A[0], 1000)
# Add start and end
line_coords_SD = [S] + [(lat, lon) for lon, lat in interm_SD] + [D]
line_coords_SA = [S] + [(lat, lon) for lon, lat in interm_SA] + [A]
line_coords_DA = [D] + [(lat, lon) for lon, lat in interm_DA] + [A]

# Plot on map
m = pu.folium_polyline(line_coords_SD, {"line_color": "green", "weight": 5})
m = pu.folium_add_polyline(m, line_coords_SA, {"line_color": "red", "weight": 5})
m = pu.folium_add_polyline(m, line_coords_DA, {"line_color": "red", "weight": 5})
m = pu.folium_add_points(m, [S, D], {"icon_color": "green"})
m = pu.folium_add_points(m, [A], {"icon_color": "red"})

m

In [ ]:
# San Francisco: Victim (D)
# New York: Peer (S)
# Amsterdam: Attacker (A)
S = (37.773972, -122.431297)
D = (40.730610, -73.935242)
threat_countries = ["Netherlands", "Brazil", "China"]

m = pu.folium_points([S, D], {"icon_color": "green"})

threat_areas = [poly_latlon(gdf_sorted[gdf_sorted["Country"] == country]["Geometry"].iloc[0].geoms[0])
                    for country in threat_countries]
threat_areas[-1] = move_left(threat_areas[-1]) # China

for area in threat_areas:
    folium.Polygon(
        locations=area,
        color="red",         # outline color
        weight=2,            # outline width
        fill=True,
        fill_color="red",   # your fill color
        fill_opacity=0.5
    ).add_to(m)

m